In [0]:
dbutils.widgets.text("business_date", "", "Business Date (YYYY-MM-DD, empty=latest)")

import sys
sys.path.append("/Workspace/Shared")
import aws_utils

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, lit, when, concat, split, element_at

spark = SparkSession.builder.getOrCreate()

env = aws_utils.get_env_prefix()

business_date_param = dbutils.widgets.get("business_date").strip()

if business_date_param:
    run_date = business_date_param
else:
    run_date = str(
        spark.table(f"`{env}-cdh`.primary.exposure")
        .filter(col("SourceSystem") == "profits")
        .agg({"BusinessDate": "max"})
        .collect()[0][0]
    )

run_date_lit = lit(run_date).cast("date")

In [0]:
# ==================== Source 1: EntityRelationships (Master-Slave + WDN) ====================

entity_rels = (
    spark.table(f"`{env}-cdh`.primary.entityrelationships")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & (col("EntityRelationshipType") == "Master - Slave Rltnp")
        & (col("EntityType") == "Exposure")
    )
    .select(
        col("EntityKey").alias("AssetKey"),
        col("RelatedEntityKey").alias("RelatedAssetKey"),
        when(
            col("EntityRelationshipSubType") == "WDN",
            lit("Restructured Loan - Writeoff Rltnp")
        ).otherwise(lit("Master - Slave Rltnp")).alias("Relation"),
    )
)

# ==================== Source 2: Restructuring (Old-New Loan relationships) ====================

restructuring = (
    spark.table(f"`{env}-cdh`.primary.restructuring")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & col("SourceSystem").isin("profits", "extfile", "arotron")
        & col("OldExposureKey").isNotNull()
        & col("NewExposureKey").isNotNull()
    )
    .select(
        col("OldExposureKey").alias("AssetKey"),
        col("NewExposureKey").alias("RelatedAssetKey"),
        lit("Restructuring: Old - New Loan").alias("Relation"), 
    )
)

subsidy_rels = (
    spark.table(f"`{env}-cdh`.landed.profits_lns_ext_subsidy_hd")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & ((col("DEACTIVATION_CODE").isNull()) | (col("DEACTIVATION_CODE") == 0))
        & col("ACCOUNT_NUMBER").isNotNull()
        & col("DEBIT_OPEN_ACCOUNT").isNotNull()
        & (col("ACCOUNT_NUMBER") != "")
        & (col("DEBIT_OPEN_ACCOUNT") != "")
    )
    .select(
        col("ACCOUNT_NUMBER").alias("AssetKey"),
        col("DEBIT_OPEN_ACCOUNT").alias("RelatedAssetKey"),
        lit("L1-L2 Subsidy Account").alias("Relation"),
    )
)

qualcoab_rels = (
    spark.table(f"`{env}-cdh`.landed.qca_fsad_old_new_accounts")
    .filter(col("BusinessDate") == run_date_lit)
    .withColumn("AssetKey",        element_at(split(col("CASE_OLD_S_ID"), "-"), -1))
    .withColumn("RelatedAssetKey", element_at(split(col("CASE_NEW_S_ID"), "-"), -1))
    .filter(
        col("AssetKey").isNotNull() & col("RelatedAssetKey").isNotNull()
        & (col("AssetKey") != "") & (col("RelatedAssetKey") != "")
    )
    .select(
        col("AssetKey"),
        col("RelatedAssetKey"),
        lit("Old-New Loan QualcoAB").alias("Relation"),
    )
)

writeoff_rels = (
    spark.table(f"`{env}-cdh`.primary.restructuring")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & col("SourceSystem").isin("profits", "extfile", "arotron")
        & col("OldExposureKey").isNotNull()
        & col("WriteOffExposureKey").isNotNull()
    )
    .select(
        col("OldExposureKey").alias("AssetKey"),
        col("WriteOffExposureKey").alias("RelatedAssetKey"),
        lit("Restructured Loan - Writeoff Rltnp").alias("Relation"),
    )
)

# ==================== Union all sources ====================

accountrelation_df = (
    entity_rels
    .unionByName(restructuring)
    .unionByName(subsidy_rels)
    .unionByName(qualcoab_rels)
    .unionByName(writeoff_rels)
    .withColumn("datadate", run_date_lit)
    .distinct()
)

# Per-source row counts
for label, df in [
    ("EntityRelationships (Master-Slave + WDN)", entity_rels),
    ("Restructuring: Old-New Loan",               restructuring),
    ("L1-L2 Subsidy Account",                     subsidy_rels),
    ("Old-New Loan QualcoAB",                     qualcoab_rels),
    ("Restructured Loan - Writeoff Rltnp (WO)",   writeoff_rels),
]:
    print(f"  {label}: {df.count():,}")
print(f"Total (after distinct): {accountrelation_df.count():,}")

In [0]:
# ==================== Write dtransfer_accountrelation ====================

target_accountrelation = f"`{env}-cdh-delivery`.it.dtransfer_accountrelation"

(
    accountrelation_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_accountrelation)
)

print(f"Written to {target_accountrelation}")